In [ ]:


import sys
from pathlib import Path
import warnings
import shutil
import pandas as pd

warnings.filterwarnings("ignore", message="X does not have valid feature names")


ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.tabsyn.models import TabSyn


dataset_name = "car"
raw_csv  = ROOT / "raw_data" / f"{dataset_name}.csv"
disc_csv = ROOT / "discretized_data" / f"{dataset_name}.csv"
disc_csv.parent.mkdir(parents=True, exist_ok=True)

print(f"Discretizing {raw_csv} -> {disc_csv}")
discretize_preprocess(str(raw_csv), str(disc_csv))

input_csv      = str(disc_csv)
output_dir     = str(ROOT / "sample_data" / dataset_name)
real_test_dir  = output_dir

# keep EXACT name "tabsyn"
synthetic_dir  = str(ROOT / "synthetic" / dataset_name / "tabsyn")


tabsyn_dir = Path(synthetic_dir)
if tabsyn_dir.exists():
    shutil.rmtree(tabsyn_dir)
tabsyn_dir.mkdir(parents=True, exist_ok=True)
print(" Cleaned synthetic folder:", tabsyn_dir)

pipeline = TrainTestSplitPipeline(
    model=lambda: TabSyn(
        d_token=16,
        decoder_epochs=200,
        diffusion_epochs=600,
        diffusion_steps=120,
        decoder_batch_size=512,
        diffusion_batch_size=512,
        lr=3e-4,
        weight_decay=1e-4,
        patience=60,
        seed=42,
        device="auto"
    )
)

try:
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
        size_category="small",
    )
    print("\n Training + TSTR completed")
    print(" Results auto-saved to:", ROOT / "Results" / dataset_name)
    print(result)

except Exception as e:
    print("\n Pipeline crashed:", repr(e))

  
    y_path = Path(synthetic_dir) / "y_synth.csv"
    if y_path.exists():
        y = pd.read_csv(y_path).iloc[:, 0]
        print("\n y_synth unique labels:", sorted(y.unique()))
        print("y_synth counts:\n", y.value_counts())
    else:
        print("\n y_synth.csv not found (TabSyn may have failed before saving).")


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\car.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\car.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\car.csv
 Cleaned synthetic folder: C:\Users\Prabu\Downloads\Katabatic\synthetic\car\tabsyn
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)


[diffusion] epoch 510/600:  33%|███▎      | 1/3 [00:00<00:00,  6.52it/s, loss=0.0854]

In [2]:
import pandas as pd

rows = [
    {"Model": "LR",      "Metric": "Accuracy", "Value": 0.6994},
    {"Model": "LR",      "Metric": "F1 Score", "Value": 0.5757},

    {"Model": "MLP",     "Metric": "Accuracy", "Value": 0.6994},
    {"Model": "MLP",     "Metric": "F1 Score", "Value": 0.5757},

    {"Model": "RF",      "Metric": "Accuracy", "Value": 0.6676},
    {"Model": "RF",      "Metric": "F1 Score", "Value": 0.6039},

    {"Model": "XGBoost", "Metric": "Accuracy", "Value": 0.6532},
    {"Model": "XGBoost", "Metric": "F1 Score", "Value": 0.6271},
]

df = pd.DataFrame(rows)

save_path = r"C:\Users\Prabu\Downloads\tabsyn_car_tstr.csv"
df.to_csv(save_path, index=False)

print("Saved CSV to:", save_path)
df


Saved CSV to: C:\Users\Prabu\Downloads\tabsyn_car_tstr.csv


,Model,Metric,Value
0,LR,Accuracy,0.6994
1,LR,F1 Score,0.5757
2,MLP,Accuracy,0.6994
3,MLP,F1 Score,0.5757
4,RF,Accuracy,0.6676
5,RF,F1 Score,0.6039
6,XGBoost,Accuracy,0.6532
7,XGBoost,F1 Score,0.6271
